## Maintenance Decision Optimization using PuLP and Google OR-Tools

This notebook demonstrates a prescriptive analytics example for reliability and maintenance.


### 📖 Problem Description

A maintenance manager must schedule preventive maintenance for four machines over a four-day planning horizon.

Each machine must be maintained **exactly once**. Only **one maintenance crew** is available per day.

The objective is to minimize the total expected maintenance cost:

$
\min \sum_{i\in I}\sum_{t\in T}
\left(C_i^m+p_{it}C_i^f\right)x_{it}
$

where:

#### Parameters

- $I$: set of machines
- $T$: set of planning days
- $C_i^m$: maintenance cost
- $C_i^f$: failure cost
- $p_{it}$: predicted probability of failure if maintenance is delayed until day $t$

#### Decision Variable

$
x_{it}=
\begin{cases}
1,&\text{if machine }i\text{ is maintained on day }t\\
0,&\text{otherwise}
\end{cases}
$

#### Constraints

Each machine is maintained exactly once

$
\sum_{t}x_{it}=1
$

Crew availability

$
\sum_i x_{it}\le1,\quad\forall t
$

Binary variables

$
x_{it}\in\{0,1\}
$

The failure probabilities are assumed to come from a predictive analytics model. The optimization model prescribes the optimal maintenance schedule.

In [ ]:
# -----------------------------
# Problem data
# -----------------------------
machines = ['M1','M2','M3','M4']
days = [1,2,3,4]

maintenance_cost = {
    'M1':200,
    'M2':300,
    'M3':250,
    'M4':350
}

failure_cost = {
    'M1':5000,
    'M2':4000,
    'M3':6000,
    'M4':7000
}

failure_prob = {
('M1',1):0.05, ('M1',2):0.08, ('M1',3):0.15, ('M1',4):0.25,
('M2',1):0.02, ('M2',2):0.03, ('M2',3):0.07, ('M2',4):0.10,
('M3',1):0.04, ('M3',2):0.05, ('M3',3):0.08, ('M3',4):0.12,
('M4',1):0.03, ('M4',2):0.05, ('M4',3):0.08, ('M4',4):0.15
}
print("Data loaded.")

Data loaded.


### 1️⃣ Solution 1 — PuLP (Mixed Integer Linear Programming)

In [4]:
# Install if needed:
# !pip install pulp

from pulp import *

model = LpProblem("MaintenanceScheduling", LpMinimize)

# Binary decision variables
x = LpVariable.dicts(
    "Maintain",
    [(i,t) for i in machines for t in days],
    cat="Binary"
)

# Objective function
model += lpSum(
    (maintenance_cost[i] + failure_prob[(i,t)]*failure_cost[i])*x[(i,t)]
    for i in machines for t in days
)

# Each machine maintained once
for i in machines:
    model += lpSum(x[(i,t)] for t in days) == 1

# One crew available each day
for t in days:
    model += lpSum(x[(i,t)] for i in machines) <= 1

model.solve()

print("Status:", LpStatus[model.status])
print("\nOptimal schedule:")
for i in machines:
    for t in days:
        if value(x[(i,t)]) > 0.5:
            print(f"{i} -> Day {t}")

print("\nOptimal objective =", value(model.objective))


Status: Optimal

Optimal schedule:
M1 -> Day 1
M2 -> Day 4
M3 -> Day 3
M4 -> Day 2

Optimal objective = 2580.0


### 2️⃣ Solution 2 — Google OR-Tools (CP-SAT)

In [5]:

# Install if needed:
# !pip install ortools

from ortools.sat.python import cp_model

cp = cp_model.CpModel()

x = {}
for i in machines:
    for t in days:
        x[(i,t)] = cp.NewBoolVar(f"{i}_{t}")

# Constraints
for i in machines:
    cp.Add(sum(x[(i,t)] for t in days) == 1)

for t in days:
    cp.Add(sum(x[(i,t)] for i in machines) <= 1)

# CP-SAT requires integer objective coefficients.
objective = []
scale = 1000

for i in machines:
    for t in days:
        coeff = int((maintenance_cost[i] +
                     failure_prob[(i,t)]*failure_cost[i]) * scale)
        objective.append(coeff * x[(i,t)])

cp.Minimize(sum(objective))

solver = cp_model.CpSolver()
status = solver.Solve(cp)

print("Status:", solver.StatusName(status))
print("\nOptimal schedule:")
for i in machines:
    for t in days:
        if solver.Value(x[(i,t)]):
            print(f"{i} -> Day {t}")

print("\nScaled objective =", solver.ObjectiveValue())
print("Approximate true objective =", solver.ObjectiveValue()/scale)


load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\zlib1.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\abseil_dll.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\utf8_validity.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\re2.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\libprotobuf.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\highs.dll...
load c:\Users\alaa.rashwan\.conda\envs\agents\Lib\site-packages\ortools\.libs\ortools.dll...
Status: OPTIMAL

Optimal schedule:
M1 -> Day 1
M2 -> Day 4
M3 -> Day 3
M4 -> Day 2

Scaled objective = 2580000.0
Approximate true objective = 2580.0


### Discussion

This notebook illustrates the workflow of **prescriptive analytics**:

1. A predictive model estimates the probability of failure for each machine.
2. An optimization model uses those predictions together with maintenance costs and resource constraints.
3. The resulting schedule minimizes the expected maintenance and failure costs while respecting crew availability.

Possible extensions include:
- multiple maintenance crews,
- maintenance durations longer than one day,
- production constraints,
- spare-parts availability,
- Remaining Useful Life (RUL)-based probabilities,
- stochastic or robust optimization,
- multi-objective optimization (cost, reliability, availability).
